# 4. LLM Inference

## Mô hình hỗ trợ
| Model | Provider | Backend |
|---|---|---|
| `gpt-4.1-nano` | OpenAI | REST API |
| `gemini-2.5-flash-lite` | Google Vertex AI | ADC |
| `llama3.1:8b` | Local | Ollama (GPU) |


<!-- | `qwen3.5:9b` | Local | Ollama (GPU) | -->

## Prompt strategies
- **Zero-shot** : chỉ mô tả task + output format
- **Few-shot** : 2 ví dụ minh hoạ input→output trước khi hỏi
- **CoT** : yêu cầu model suy luận từng bước trước khi ra JSON
- **Retrival-Aware**: trích xuất dữ liệu từ các nguồn bên ngoài để cung cấp thông tin chính xác

## Cài đặt thư viện + Repo + LLM

In [1]:
!pip install openai anthropic google-genai requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 55.9 MB/s eta 0:00:00


In [2]:
!pip install colab-xterm
%load_ext colabxterm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 11.2 MB/s eta 0:00:00


In [3]:
!apt-get update -qq
!apt-get install -y zstd
!apt-get install -y lshw pciutils   # để Ollama detect GPU
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,770 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lists... Done
Building dependency tree... 

In [4]:
# Xác nhận Ollama thấy GPU
!ollama info 2>/dev/null | grep -i gpu || echo "GPU not detected"

# Kiểm tra CUDA từ phía Python
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "nvidia-smi not found")

GPU not detected
Wed Jun  3 08:15:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------------

In [5]:
!ollama --version

In [6]:
!git clone https://github.com/DuongThai2712/smart-tourism

Cloning into 'smart-tourism'...
remote: Enumerating objects: 1411, done.
remote: Counting objects: 100% (1411/1411), done.
remote: Compressing objects: 100% (1276/1276), done.
remote: Total 1411 (delta 436), reused 1068 (delta 99), pack-reused 0 (from 0)
Receiving objects: 100% (1411/1411), 17.68 MiB | 13.79 MiB/s, done.
Resolving deltas: 100% (436/436), done.


## Cấu hình API keys & tham số toàn cục

In [7]:
!gcloud auth application-default login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=tvgo4MzHAPw2TPSM5bf1iKvMPEHfrZ&prompt=consent&token_usage=remote&access_type=offline&code_challenge=DnBucNW3CoM-WFYBmmnLf9KljfUGNHd5LnG9MY3185I&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AeoWuM9A7CaC1PmZf0N2XJNl9D87jAiDQUK8GomOO5L7GoUn6xRX6TY54utRgCZrnw_WkQ

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [8]:
!gcloud auth application-default set-quota-project gemini-498219


Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "gemini-498219" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [9]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)

print("Ollama started")
!ollama ps

Ollama started
NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


In [10]:
import os
# OLLAMA local
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434"
)

try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/chat", timeout=5)
    OLLAMA_AVAILABLE = r.status_code == 200
except Exception as e:
    print("Ollama unavailable:", e)
    OLLAMA_AVAILABLE = False

In [11]:
!ollama pull llama3.1:8b

In [12]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


In [13]:
#kiểm tra xem llm có chạy không?
!ollama run llama3.1:8b "hello" && ollama ps

Hello! How are you today? Is there something I can help you with or would y
you like to chat?

NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.1:8b    46e0c10c039e    9.2 GB    100% GPU     32768      4 minutes from now    


In [ ]:
import torch
import random
#  API keys (ưu tiên biến môi trường, fallback điền trực tiếp) ─
# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

OPENAI_KEY = os.getenv("OPENAI_API_KEY", "")

#  Google Vertex AI — dùng ADC (Application Default Credentials) ─
VERTEX_PROJECT  = "gemini-498219"
VERTEX_LOCATION = "global"

#  GPU detection ─
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU: {GPU_NAME} ({GPU_MEM} GB VRAM)")
else:
    print("GPU: không có → Ollama dùng CPU")

#  Tham số inference
MAX_TOKENS = 768   # tăng lên cho CoT
TEMPERATURE = 0.2
BATCH_SIZE = 16      # số context xử lý song song mỗi lần
MAX_RETRIES = 3
DELAY_BETWEEN_CALLS = 0.8   # giây giữa các API call (tránh rate limit)

PROMPT_STRATEGIES = ["zero_shot", "few_shot", "cot", "retrieval_aware"]

MODELS_CONFIG = {
    # "qwen3.5:9b":            {"backend": "ollama",     "enabled": True},
    "llama3.1:8b":           {"backend": "ollama",     "enabled": True},
    "gpt-5.4-nano":          {"backend": "openai",     "enabled": bool(OPENAI_KEY)},
    "gemini-3.1-flash-lite": {"backend": "vertex",     "enabled": True},
}

print("\nTrạng thái models:")
for m, cfg in MODELS_CONFIG.items():
    status = "✓ enabled" if cfg["enabled"] else "✗ disabled (thiếu key)"
    print(f"  [{cfg['backend']:12s}] {m:30s} {status}")

GPU: NVIDIA A100-SXM4-40GB (39 GB VRAM)

Trạng thái models:
  [ollama      ] llama3.1:8b                    ✓ enabled
  [openai      ] gpt-5.4-nano                   ✓ enabled
  [vertex      ] gemini-3.1-flash-lite          ✓ enabled


## Load context

In [15]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path("/content/smart-tourism")

rag_path    = project_root / "Datasets" / "RAG"
raw_path    = project_root / "Datasets" / "Raw"
result_path = project_root / "Datasets" / "Results"
result_path.mkdir(parents=True, exist_ok=True)


In [16]:
with open(rag_path / "llm_contexts.json", encoding="utf-8") as f:
    all_contexts = json.load(f)

print(f"Đã load {len(all_contexts)} contexts")
print(f"Cities : {sorted(set(c['city'] for c in all_contexts))}")

ctx0 = all_contexts[0]
print(f"\nVí dụ context đầu tiên:")
print(f"  user_id         :{ctx0['user_id']}")
print(f"  city            : {ctx0['city']}")
print(f"  budget_minutes  : {ctx0['budget_minutes']}")
print(f"  retrieved_pois  : {len(ctx0['retrieved_pois'])} POI")

Đã load 11323 contexts
Cities : ['Athens', 'Barcelona', 'Budapest', 'Edinburgh', 'Glasgow', 'London', 'Madrid', 'Melbourne', 'NewDelhi', 'Osaka', 'Perth', 'Toronto', 'Vienna']

Ví dụ context đầu tiên:
  user_id         :100721644@N08
  city            : Athens
  budget_minutes  : 480
  retrieved_pois  : 20 POI


## Prompt Strategies

Mỗi strategy gồm `system_prompt` + hàm `build_user_prompt(ctx)`.  
Tất cả đều yêu cầu output JSON cùng schema để parse thống nhất.

In [17]:
OUTPUT_SCHEMA = """\
Return ONLY a valid JSON object with this exact schema (no markdown, no extra text):
{"itinerary": ["POI Name 1", "POI Name 2", ...], "total_time_min": <integer>, "reason": "<brief>"}"""


def _profile_block(ctx: dict) -> str:
    """Shared tourist profile block."""
    visited  = ctx["user_profile"]["visited_pois"]
    top_cats = ctx["user_profile"]["top_categories"]
    visited_str = ", ".join(visited) if visited else "None yet"
    cat_lines = "\n".join(
        f"  - {c['poiTheme']}: interest {c['int_u_c']:.2f}"
        for c in top_cats
    ) if top_cats else "  - No strong preferences detected"
    return (
        f"City: {ctx['city']}\n"
        f"Time budget: {ctx['budget_minutes']} minutes\n"
        f"Previously visited: {visited_str}\n"
        f"Interest categories:\n{cat_lines}"
    )


def _poi_block(ctx: dict) -> str:
    """Numbered POI list block."""
    return "\n".join(
        f"  {i+1:2d}. "
        f"[{p['category']:12s}] "
        f"{p['name']:35s} "
        f"visit={p['avg_visit_min']:3.0f}min "
        f"match_score={p['similarity']:.3f}"
        for i, p in enumerate(ctx["retrieved_pois"])
    )



In [18]:
# Rule chung cho cả 4 prompt
COMMON_RULES = """
Rules:
- Choose 3-6 POIs only from input
- Stay within time budget
- Prefer high-interest/high-match POIs
- Avoid visited POIs if possible
- Use most available time
"""

# STRATEGY 1 — ZERO-SHOT
ZERO_SHOT_SYSTEM = f"""
Plan a 1-day itinerary.
{COMMON_RULES}
Return JSON only.
{OUTPUT_SCHEMA}
"""


def zero_shot_user(ctx: dict) -> str:
    return (
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs:\n{_poi_block(ctx)}\n\n"
        f"Generate the optimal itinerary."
    )



# STRATEGY 2 — FEW-SHOT (2 examples)
FEW_SHOT_EXAMPLES = """\
=== EXAMPLE 1 ===
Tourist profile:
  City: Athens
  Budget: 300 min
  Interests: Museum (3.2), Historical (2.8)
Available POIs:
  1. [Museum] Acropolis Museum visit=90min match_score=0.94
  2. [Historical] Parthenon visit=60min match_score=0.97
  3. [Shopping] Flea Market visit=45min match_score=0.35
  4. [Park] National Garden visit=30min match_score=0.28
  5. [Historical] Temple of Olympian Zeus visit=40min match_score=0.88
Output:
{"itinerary": ["Parthenon", "Acropolis Museum", "Temple of Olympian Zeus", "Monastiraki Flea Market"], "total_time_min": 255, "reason": "Cluster historical sites first, then museum nearby, end at market. High match scores and strong historical theme."}

=== EXAMPLE 2 ===
Tourist profile:
  City: Barcelona
  Budget: 240 min
  Interests: Architecture (4.1), Art (2.5)
Available POIs:
  1. [Architecture] Sagrada Familia visit=90min match_score=0.98
  2. [Art] Picasso Museum visit=60min match_score=0.83
  3. [Architecture] Casa Batllo visit=45min match_score=0.92
  4. [Beach] Barceloneta Beach visit=60min match_score=0.31
  5. [Park] Park Guell visit=50min match_score=0.86
Output:
{"itinerary": ["Sagrada Familia", "Casa Batllo", "Picasso Museum"], "total_time_min": 195, "reason": "Architecture focus, grouped on Eixample then Gothic Quarter. Prioritized architecture and art attractions with the highest match scores."}

=== YOUR TASK ==="""

FEW_SHOT_SYSTEM = f"""
You are an expert one-day tourism itinerary planner.
{COMMON_RULES}
Return JSON only.
{OUTPUT_SCHEMA}
"""


def few_shot_user(ctx: dict) -> str:
    return (
        f"{FEW_SHOT_EXAMPLES}\n"
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs:\n{_poi_block(ctx)}\n\n"
        f"Output:"
    )



# STRATEGY 3 — CHAIN-OF-THOUGHT (CoT)
COT_SYSTEM = f"""
You are an expert one-day tourism itinerary planner. Think step by step:
Step 1: Identify tourists' strongest interests.
Step 2: Rank POIs by category relevance and match_score.
Step 3: Select a subset maximizing relevance while staying within budget.
Step 4: Remove unnecessary POIs.
Step 5: Order POIs by logical thematic flow.
Step 6: Output the final JSON.

{COMMON_RULES}

Write reasoning inside a <thinking> block then return JSON only.
{OUTPUT_SCHEMA}
"""

def cot_user(ctx):
    return (
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs:\n{_poi_block(ctx)}\n\n"
        f"Think step by step."
    )

# STRATEGY 4 — RETRIEVAL-AWARE
RETRIEVAL_AWARE_SYSTEM = f"""
You are an expert tourism itinerary planner. The POIs below were retrieved by a recommendation system. Each POI contains a match_score.
Definition:
- match_score ranges from 0 to 1.
- Higher match_score means the POI is more relevant to the tourist's interests.
- POIs with higher match_score should generally be preferred.

{COMMON_RULES}

When two POIs are similar, prefer the one with the higher match_score.

Return JSON only.
{OUTPUT_SCHEMA}
"""
def retrieval_aware_user(ctx):
    return (
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Retrieved candidate POIs:\n{_poi_block(ctx)}\n\n"
        f"Generate the itinerary that maximizes overall relevance."
    )

In [19]:
PROMPT_REGISTRY = {
    "zero_shot": {
        "system": ZERO_SHOT_SYSTEM,
        "user_fn": zero_shot_user,
    },
    "few_shot": {
        "system": FEW_SHOT_SYSTEM,
        "user_fn": few_shot_user,
    },
    "cot": {
        "system": COT_SYSTEM,
        "user_fn": cot_user,
    },
    "retrieval_aware": {
        "system": RETRIEVAL_AWARE_SYSTEM,
        "user_fn": retrieval_aware_user,
    },
}

print("4 prompt strategies đã sẵn sàng: zero_shot | few_shot | cot | retrieval-aware")

#  Kiểm tra nhanh
for strat, reg in PROMPT_REGISTRY.items():
    user_msg = reg["user_fn"](ctx0)
    print(f"\n[{strat}] system={len(reg['system'])} chars | user_msg={len(user_msg)} chars")
    print(f"  User preview: {user_msg[:120].strip()} ...")

4 prompt strategies đã sẵn sàng: zero_shot | few_shot | cot | retrieval-aware

[zero_shot] system=394 chars | user_msg=2015 chars
  User preview: Tourist profile:
City: Athens
Time budget: 480 minutes
Previously visited: Museum of Illusions Athens - Greece
Interest ...

[few_shot] system=424 chars | user_msg=3374 chars
  User preview: === EXAMPLE 1 ===
Tourist profile:
  City: Athens
  Budget: 300 min
  Interests: Museum (3.2), Historical (2.8)
Availabl ...

[cot] system=780 chars | user_msg=2003 chars
  User preview: Tourist profile:
City: Athens
Time budget: 480 minutes
Previously visited: Museum of Illusions Athens - Greece
Interest ...

[retrieval_aware] system=768 chars | user_msg=2050 chars
  User preview: Tourist profile:
City: Athens
Time budget: 480 minutes
Previously visited: Museum of Illusions Athens - Greece
Interest ...


## Response Parser

Parse JSON từ response LLM, xử lý các trường hợp:
- Markdown fences ```json ... ```
- <thinking>...</thinking> tags (CoT / extended thinking)
- Text thừa trước/sau JSON

In [20]:
import re

def parse_llm_response(raw: str) -> dict:
    if not raw:
        return {"itinerary": [], "total_time_min": 0, "reason": "empty_response"}

    # Loại bỏ thinking tags (CoT responses)
    text = re.sub(r"<thinking>.*?</thinking>", "", raw, flags=re.DOTALL)
    # Loại bỏ markdown fences
    text = re.sub(r"```(?:json)?\s*", "", text)
    text = text.strip()

    # Tìm JSON object đầu tiên
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        # Thử tìm JSON lồng nhau
        match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            # Đảm bảo có đủ các key cần thiết
            parsed.setdefault("itinerary", [])
            parsed.setdefault("total_time_min", 0)
            parsed.setdefault("reason", "")
            # Đảm bảo itinerary là list of strings
            if not isinstance(parsed["itinerary"], list):
                parsed["itinerary"] = []
            return parsed
        except json.JSONDecodeError:
            pass

    return {
        "itinerary": [],
        "total_time_min": 0,
        "reason": "parse_error",
        "raw_truncated": raw[:300],
    }

In [21]:
# Test parser
test_cases = [
    '{"itinerary": ["A"], "total_time_min": 60, "reason": "ok"}',
    'abc {"itinerary": ["B"], "total_time_min": 90, "reason": "ok"} xyz',
    '{"itinerary": ["C"]}',  # thiếu key
    '{"itinerary": "C"}',    # sai kiểu dữ liệu
]
for tc in test_cases:
    result = parse_llm_response(tc)
    status = "✓" if result["itinerary"] else "✗"
    print(f"{status} itinerary={result['itinerary']} | reason={result['reason']}")

✓ itinerary=['A'] | reason=ok
✓ itinerary=['B'] | reason=ok
✓ itinerary=['C'] | reason=
✗ itinerary=[] | reason=


## LLM Backend Wrappers

In [22]:
import time
import requests


def _make_result(model, strategy, raw_out, latency, prompt_tok, completion_tok):
    """Tạo dict kết quả chuẩn từ raw LLM output."""
    parsed = parse_llm_response(raw_out)
    parsed.update({
        "model":             model,
        "prompt_strategy":   strategy,
        "latency_sec":       round(latency, 3),
        "prompt_tokens":     prompt_tok,
        "completion_tokens": completion_tok,
    })
    return parsed



#  2. OpenAI — GPT-5.4-nano
def call_openai(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)
    t0 = time.time()
    resp = client.chat.completions.create(
        model       = model_id,
        max_completion_tokens  = MAX_TOKENS,
        temperature = TEMPERATURE,
        messages    = [
            {"role": "system", "content": system},
            {"role": "user",   "content": user_msg},
        ],
    )
    return _make_result(
        model_id, strategy,
        raw_out        = resp.choices[0].message.content,
        latency        = time.time() - t0,
        prompt_tok     = resp.usage.prompt_tokens,
        completion_tok = resp.usage.completion_tokens,
    )


#  3. Google Vertex AI — Gemini 3.1 Flash Lite (ADC)
_vertex_client = None

def _get_vertex_client():
    global _vertex_client
    if _vertex_client is None:
        from google import genai
        _vertex_client = genai.Client(
            vertexai=True,
            project=VERTEX_PROJECT,
            location=VERTEX_LOCATION,
        )
    return _vertex_client


def call_vertex(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    from google.genai import types as gtypes
    client = _get_vertex_client()
    t0 = time.time()
    resp = client.models.generate_content(
        model    = model_id,
        contents = user_msg,
        config   = gtypes.GenerateContentConfig(
            system_instruction = system,
            max_output_tokens  = MAX_TOKENS,
            temperature        = TEMPERATURE,
        ),
    )
    usage = resp.usage_metadata
    return _make_result(
        model_id, strategy,
        raw_out        = resp.text,
        latency        = time.time() - t0,
        prompt_tok     = getattr(usage, "prompt_token_count",     0) or 0,
        completion_tok = getattr(usage, "candidates_token_count", 0) or 0,
    )


#  4. Ollama — local models (qwen3.5:9b / llama3.1:8b)
def _ollama_available() -> bool:
    """Kiểm tra Ollama server có đang chạy không."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False


def call_ollama(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    """
    Gọi Ollama REST API (/api/chat).
    Ollama tự detect GPU nếu model đã được pull và CUDA/ROCm khả dụng.
    Dùng num_gpu=-1 để load toàn bộ layers lên GPU.
    """
    payload = {
        "model": model_id,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": 512,     # giảm cho local
            "num_gpu": 99 if DEVICE == "cuda" else 0,  # -1 = all layers on GPU
        },
        "messages": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": user_msg},
        ],
    }
    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=600,
    )
    resp.raise_for_status()
    data = resp.json()
    raw_text   = data.get("message", {}).get("content", "")
    usage      = data.get("prompt_eval_count", 0)
    completion = data.get("eval_count", 0)
    return _make_result(
        model_id, strategy,
        raw_out        = raw_text,
        latency        = time.time() - t0,
        prompt_tok     = usage,
        completion_tok = completion,
    )


In [23]:
#  Dispatch table
BACKEND_FN = {
    "openai":    call_openai,
    "vertex":    call_vertex,
    "ollama":    call_ollama,
}


def call_model(model_name: str, strategy: str, ctx: dict) -> dict:
    """
    Gọi model với strategy chỉ định, có retry + exponential backoff.
    Trả về dict kết quả chuẩn.
    """
    cfg      = MODELS_CONFIG[model_name]
    backend  = cfg["backend"]
    reg      = PROMPT_REGISTRY[strategy]
    system   = reg["system"]
    user_msg = reg["user_fn"](ctx)
    fn       = BACKEND_FN[backend]

    for attempt in range(MAX_RETRIES):
        try:
            result = fn(model_name, system, user_msg, strategy)
            result["user_id"] = ctx["user_id"]
            result["city"]    = ctx["city"]
            return result
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [{model_name}/{strategy}] attempt {attempt+1} failed: {e} — retry in {wait}s")
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)

    # Trả về empty result sau khi hết retry
    return {
        "model": model_name, "prompt_strategy": strategy,
        "user_id": ctx["user_id"], "city": ctx["city"],
        "itinerary": [], "total_time_min": 0, "reason": "max_retries_exceeded",
        "latency_sec": 0, "prompt_tokens": 0, "completion_tokens": 0,
    }


#  Kiểm tra Ollama ─
ollama_ok = _ollama_available()
print(f"Ollama server ({OLLAMA_BASE_URL}): {'✓ online' if ollama_ok else '✗ offline'}")
if not ollama_ok:
    print("  → Các model ollama sẽ bị skip. Chạy: ollama serve")
    for m, cfg in MODELS_CONFIG.items():
        if cfg["backend"] == "ollama":
            cfg["enabled"] = False

print("\nBackend wrappers đã sẵn sàng.")

Ollama server (http://127.0.0.1:11434): ✓ online

Backend wrappers đã sẵn sàng.


## Metrics

In [24]:
def normalize_poi(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def compute_itinerary_metrics(predicted: list, actual: list) -> dict:
    """Recall, Precision, F-score (Section 7.2 bài báo)."""
    pred_set   = set(normalize_poi(p) for p in predicted)
    actual_set = set(normalize_poi(a) for a in actual)
    if not actual_set or not pred_set:
        return {"recall": 0.0, "precision": 0.0, "f_score": 0.0}
    tp        = len(pred_set & actual_set)
    recall    = tp / len(actual_set)
    precision = tp / len(pred_set)
    f_score   = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "recall":    round(recall,    4),
        "precision": round(precision, 4),
        "f_score":   round(f_score,   4),
    }


def compute_ae_ue(itinerary: list, profit_per_poi: dict, max_profit: float = 1.0) -> dict:
    """AE, UE (Eq. 22, 23 bài báo — simplified for LLM output)."""
    if not itinerary:
        return {"ae": 0.0, "ue": 0.0}
    profits = [profit_per_poi.get(normalize_poi(p), 0.0) for p in itinerary]
    ae = sum(profits) / (len(profits) * max_profit) if max_profit > 0 else 0.0
    ue = ae * (sum(profits) / max(max_profit * len(profits), 1e-9))
    return {"ae": round(ae, 4), "ue": round(ue, 4)}


print("Metrics functions ready.")

Metrics functions ready.


## Load Ground Truth & Profit Map

In [25]:
#  Load visit và POI data ─
visit_frames, poi_frames = [], []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    city = city_dir.name
    for fname, store in [("touristsVisits.csv", visit_frames), ("POIs.csv", poi_frames)]:
        fp = city_dir / fname
        if fp.exists():
            df = pd.read_csv(fp)
            df["city"] = city
            store.append(df)

visit_pdf = pd.concat(visit_frames, ignore_index=True)
poi_pdf   = pd.concat(poi_frames,   ignore_index=True)

#  Profit proxy = normalized visit count ─
pop = (
    visit_pdf.groupby(["city", "poiID"])["userID"].count()
    .reset_index(name="pop")
    .merge(poi_pdf[["city", "poiID", "poiName"]], on=["city", "poiID"])
)
city_max    = pop.groupby("city")["pop"].max().rename("pop_max")
pop         = pop.join(city_max, on="city")
pop["profit_norm"] = pop["pop"] / pop["pop_max"]

# Dict (city, poi_normalized) → profit
profit_map = {
    (r["city"], normalize_poi(r["poiName"])): r["profit_norm"]
    for _, r in pop.iterrows()
}

# Dict user_id → actual sequence (sorted by timestamp)
actual_map = (
    visit_pdf
    .merge(poi_pdf[["city", "poiID", "poiName"]], on=["city", "poiID"])
    .sort_values("dateTaken")
    .groupby(["userID", "city"])["poiName"]
    .apply(lambda x: list(dict.fromkeys(x)))  # deduplicate giữ thứ tự
    .to_dict()
)

print(f"Profit map : {len(profit_map):,} entries")
print(f"Actual map : {len(actual_map):,} (user, city) pairs")

Profit map : 421 entries
Actual map : 11,323 (user, city) pairs


## Batch Inference Engine

Chạy inference cho 1 (model, strategy, context) tuple.
    Tính metrics ngay sau khi nhận kết quả.

Batch inference:
- Local models (ollama): ThreadPoolExecutor để chạy song song trong batch
- API models (anthropic/openai/vertex): tuần tự với delay tránh rate limit

In [26]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm


def run_single(
    model_name,
    strategy,
    ctx,
    actual_map,
    profit_by_city,
):
    result = call_model(model_name, strategy, ctx)

    itinerary = result.get("itinerary", [])

    city = ctx["city"]
    user_id = ctx["user_id"]

    actual_seq = actual_map.get((user_id, city), [])
    city_profit = profit_by_city.get(city, {})

    trad_m = compute_itinerary_metrics(
        itinerary,
        actual_seq,
    )

    alloc_m = compute_ae_ue(
        itinerary,
        city_profit,
    )

    return {
        "user_id": user_id,
        "city": city,
        "budget_min": ctx["budget_minutes"],

        "model": model_name,
        "prompt_strategy": strategy,

        "predicted_itinerary": "|".join(itinerary),
        "actual_itinerary": "|".join(actual_seq),

        "n_pois_predicted": len(itinerary),
        "n_pois_actual": len(actual_seq),

        "predicted_time_min": result.get(
            "total_time_min", 0
        ),

        "reason": result.get("reason", ""),

        "latency_sec": result.get(
            "latency_sec", 0
        ),

        "prompt_tokens": result.get(
            "prompt_tokens", 0
        ),

        "completion_tokens": result.get(
            "completion_tokens", 0
        ),

        **trad_m,
        **alloc_m,
    }



In [27]:
def run_batch_inference(
    contexts: list,
    models_config: dict,
    strategies: list,
    actual_map: dict,
    profit_map: dict,
    batch_size: int = BATCH_SIZE,
    api_delay: float = DELAY_BETWEEN_CALLS,
) -> list:

    enabled_models = [m for m, c in models_config.items() if c["enabled"]]
    total_jobs     = len(contexts) * len(enabled_models) * len(strategies)

    print(f"Số contexts sẽ chạy: {len(contexts)}")
    print(f"Tổng jobs: {total_jobs:,} "
          f"({len(contexts)} ctx × {len(enabled_models)} models × {len(strategies)} strategies)")
    print(f"Models   : {enabled_models}")
    print(f"Strategies: {strategies}")

    all_results = []
    pbar        = tqdm(total=total_jobs, desc="Inference", unit="job")

    ctx_batches = [
        contexts[i : i + batch_size]
        for i in range(0, len(contexts), batch_size)
    ]

    for model_name in enabled_models:
        backend  = models_config[model_name]["backend"]
        is_local = backend == "ollama"

        # FIX: local dùng 2 workers (tránh OOM trên Colab T4)
        # API dùng 10 workers để gọi song song
        max_workers = 2 if is_local else 10

        for strategy in strategies:
            print(f"\n {model_name} / {strategy} ")

            # FIX: tạo pool 1 lần cho cả strategy, không tạo lại mỗi batch
            with ThreadPoolExecutor(max_workers=max_workers) as pool:
                for batch_idx, batch in enumerate(ctx_batches):

                    futures = {
                        pool.submit(
                            run_single,
                            model_name,
                            strategy,
                            ctx,
                            actual_map,
                            profit_map,
                        ): ctx
                        for ctx in batch
                    }

                    for future in as_completed(futures):
                        try:
                            row = future.result()
                            all_results.append(row)
                        except Exception as e:
                            print(f"future error: {e}")
                        pbar.update(1)

                    # Delay chỉ cho API, không cho local
                    if not is_local and batch_idx < len(ctx_batches) - 1:
                        if model_name == "gemini-3.1-flash-lite":
                            time.sleep(api_delay * random.randint(2, 5))
                        else:
                            time.sleep(api_delay)

    pbar.close()
    print(f"\n✓ Hoàn thành: {len(all_results):,} / {total_jobs:,} jobs")
    return all_results


print("✓ Batch inference engine sẵn sàng.")

✓ Batch inference engine sẵn sàng.


## Chạy Inference

In [28]:
random.seed(42)
sampled_contexts = random.sample(all_contexts, 200)      # Lấy ngẫu nhiên 200 mẫu do quá nhiều

print(f"Số contexts sẽ chạy: {len(sampled_contexts)}")

all_results = run_batch_inference(
    contexts       = sampled_contexts,
    models_config  = MODELS_CONFIG,
    strategies     = PROMPT_STRATEGIES,
    actual_map     = actual_map,
    profit_map     = profit_map,
    batch_size     = BATCH_SIZE,
    api_delay      = DELAY_BETWEEN_CALLS,
)

Số contexts sẽ chạy: 200
Số contexts sẽ chạy: 200
Tổng jobs: 2,400 (200 ctx × 3 models × 4 strategies)
Models   : ['llama3.1:8b', 'gpt-5.4-nano', 'gemini-3.1-flash-lite']
Strategies: ['zero_shot', 'few_shot', 'cot', 'retrieval_aware']


Inference:   0%|          | 0/2400 [00:00<?, ?job/s]


 llama3.1:8b / zero_shot 

 llama3.1:8b / few_shot 

 llama3.1:8b / cot 

 llama3.1:8b / retrieval_aware 

 gpt-5.4-nano / zero_shot 

 gpt-5.4-nano / few_shot 

 gpt-5.4-nano / cot 

 gpt-5.4-nano / retrieval_aware 

 gemini-3.1-flash-lite / zero_shot 

 gemini-3.1-flash-lite / few_shot 

 gemini-3.1-flash-lite / cot 

 gemini-3.1-flash-lite / retrieval_aware 

✓ Hoàn thành: 2,400 / 2,400 jobs


## Lưu kết quả

In [29]:
results_df = pd.DataFrame(all_results)

# Lưu raw
results_df.to_csv(result_path / "llm_results_v2.csv", index=False)
results_df.to_json(result_path / "llm_results_v2.json", orient="records", indent=2)

print(f"✓ Đã lưu {len(results_df):,} rows")
print(f"  Columns: {list(results_df.columns)}")
results_df.head(3)

✓ Đã lưu 2,400 rows
  Columns: ['user_id', 'city', 'budget_min', 'model', 'prompt_strategy', 'predicted_itinerary', 'actual_itinerary', 'n_pois_predicted', 'n_pois_actual', 'predicted_time_min', 'reason', 'latency_sec', 'prompt_tokens', 'completion_tokens', 'recall', 'precision', 'f_score', 'ae', 'ue']


,user_id,city,budget_min,model,prompt_strategy,predicted_itinerary,actual_itinerary,n_pois_predicted,n_pois_actual,predicted_time_min,reason,latency_sec,prompt_tokens,completion_tokens,recall,precision,f_score,ae,ue
0,28962482@N02,Vienna,480,llama3.1:8b,zero_shot,St. Charles' Church Vienna|Austrian National L...,St._Stephen%27s_Cathedral_Vienna,3,1,336262,Selected POIs with highest match scores and wi...,6.208,725,67,0.0,0.0,0.0,0.0,0.0
1,7822998@N03,Budapest,480,llama3.1:8b,zero_shot,Budapest Kunsthalle|Hungarian National Museum|...,Palace_of_Arts_(Budapest)|Great_Market_Hall_(B...,3,3,360,Selected POIs with high match scores and withi...,7.009,747,65,0.0,0.0,0.0,0.0,0.0
2,13355897@N02,Barcelona,480,llama3.1:8b,zero_shot,Gaudi House Museum|History Museum of Catalonia,Museum_of_the_History_of_Barcelona,2,1,1028,The Gaudi House Museum is a high-interest POI ...,1.675,702,72,0.0,0.0,0.0,0.0,0.0


In [30]:
metrics_cols = ["recall", "precision", "f_score", "ae", "ue"]

#  Theo model × strategy
summary_model_strategy = (
    results_df
    .groupby(["model", "prompt_strategy"])[metrics_cols]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["model", "prompt_strategy"])
)

print("=" * 90)
print("BẢNG 1 — Kết quả trung bình theo Model × Prompt Strategy")
print("=" * 90)
print(summary_model_strategy.to_string(index=False))
summary_model_strategy.to_csv(result_path / "summary_model_strategy.csv", index=False)

BẢNG 1 — Kết quả trung bình theo Model × Prompt Strategy
                model prompt_strategy  recall  precision  f_score  ae  ue
gemini-3.1-flash-lite             cot  0.0000     0.0000   0.0000 0.0 0.0
gemini-3.1-flash-lite        few_shot  0.0000     0.0000   0.0000 0.0 0.0
gemini-3.1-flash-lite retrieval_aware  0.0000     0.0000   0.0000 0.0 0.0
gemini-3.1-flash-lite       zero_shot  0.0000     0.0000   0.0000 0.0 0.0
         gpt-5.4-nano             cot  0.0000     0.0000   0.0000 0.0 0.0
         gpt-5.4-nano        few_shot  0.0000     0.0000   0.0000 0.0 0.0
         gpt-5.4-nano retrieval_aware  0.0000     0.0000   0.0000 0.0 0.0
         gpt-5.4-nano       zero_shot  0.0000     0.0000   0.0000 0.0 0.0
          llama3.1:8b             cot  0.0024     0.0050   0.0032 0.0 0.0
          llama3.1:8b        few_shot  0.0000     0.0000   0.0000 0.0 0.0
          llama3.1:8b retrieval_aware  0.0025     0.0017   0.0020 0.0 0.0
          llama3.1:8b       zero_shot  0.0000     0.000

In [31]:
#  Theo model × city (strategy tốt nhất)
best_strategy_per_model = (
    summary_model_strategy
    .sort_values("f_score", ascending=False)
    .groupby("model")
    .first()
    .reset_index()[["model", "prompt_strategy", "recall", "precision", "f_score", "ae", "ue"]]
)

print("\n" + "=" * 90)
print("BẢNG 2 — Best prompt strategy per model (theo F-score)")
print("=" * 90)
print(best_strategy_per_model.to_string(index=False))

#  Theo city × model (strategy = best f-score)
summary_city = (
    results_df
    .groupby(["city", "model"])[metrics_cols]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["city", "f_score"], ascending=[True, False])
)


print("BẢNG 3 — Kết quả theo City × Model (trung bình tất cả strategies)")
print("=" * 90)
print(summary_city.to_string(index=False))
summary_city.to_csv(result_path / "summary_city_model.csv", index=False)


BẢNG 2 — Best prompt strategy per model (theo F-score)
                model prompt_strategy  recall  precision  f_score  ae  ue
gemini-3.1-flash-lite             cot  0.0000      0.000   0.0000 0.0 0.0
         gpt-5.4-nano             cot  0.0000      0.000   0.0000 0.0 0.0
          llama3.1:8b             cot  0.0024      0.005   0.0032 0.0 0.0
BẢNG 3 — Kết quả theo City × Model (trung bình tất cả strategies)
     city                 model  recall  precision  f_score  ae  ue
   Athens gemini-3.1-flash-lite  0.0000     0.0000   0.0000 0.0 0.0
   Athens          gpt-5.4-nano  0.0000     0.0000   0.0000 0.0 0.0
   Athens           llama3.1:8b  0.0000     0.0000   0.0000 0.0 0.0
Barcelona gemini-3.1-flash-lite  0.0000     0.0000   0.0000 0.0 0.0
Barcelona          gpt-5.4-nano  0.0000     0.0000   0.0000 0.0 0.0
Barcelona           llama3.1:8b  0.0000     0.0000   0.0000 0.0 0.0
 Budapest gemini-3.1-flash-lite  0.0000     0.0000   0.0000 0.0 0.0
 Budapest          gpt-5.4-nano  0.000

In [34]:
#  Chi phí token và latency ─
cost_df = (
    results_df
    .groupby(["model", "prompt_strategy"])[["latency_sec", "prompt_tokens", "completion_tokens"]]
    .mean()
    .round(2)
    .reset_index()
)
cost_df["total_tokens"] = cost_df["prompt_tokens"] + cost_df["completion_tokens"]

# Giá USD per 1M tokens (cập nhật tháng 6/2026)
PRICE = {
    "gpt-4.1-nano":          {"in": 0.20,  "out": 1.25},
    "gemini-2.5-flash-lite": {"in": 0.25, "out": 1.50},
    "qwen3.5:9b":            {"in": 0.0,   "out": 0.0},   # local
    "llama3.1:8b":           {"in": 0.0,   "out": 0.0},   # local
}


print("BẢNG 4 — Chi phí token & latency trung bình")
print("=" * 90)
print(cost_df.to_string(index=False))
cost_df.to_csv(result_path / "cost_latency_summary.csv", index=False)

print("\nTất cả output đã lưu:")
for f in sorted(result_path.iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:45s} {size_kb:6d} KB")

BẢNG 4 — Chi phí token & latency trung bình
                model prompt_strategy  latency_sec  prompt_tokens  completion_tokens  total_tokens
gemini-3.1-flash-lite             cot         1.51         880.86             138.90       1019.76
gemini-3.1-flash-lite        few_shot         1.15        1251.86              66.83       1318.69
gemini-3.1-flash-lite retrieval_aware         1.32         881.86              90.66        972.52
gemini-3.1-flash-lite       zero_shot         1.24         788.86              85.96        874.82
         gpt-5.4-nano             cot         1.89         757.76              90.60        848.36
         gpt-5.4-nano        few_shot         2.73        1097.77              92.24       1190.01
         gpt-5.4-nano retrieval_aware         1.87         755.76              90.40        846.16
         gpt-5.4-nano       zero_shot         3.43         672.76              86.92        759.68
          llama3.1:8b             cot         8.27         774.08

In [33]:
# Tổng hợp score = 0.4*f_score + 0.3*ae + 0.3*ue
ranking = summary_model_strategy.copy()
ranking["composite_score"] = (
    0.4 * ranking["f_score"] +
    0.3 * ranking["ae"] +
    0.3 * ranking["ue"]
).round(4)

print("=" * 60)
print("RANKING TỔNG HỢP (0.4×F1 + 0.3×AE + 0.3×UE)")
print("=" * 60)
print(
    ranking
    .sort_values("composite_score", ascending=False)
    [["model", "prompt_strategy", "f_score", "ae", "ue", "composite_score"]]
    .to_string(index=False)
)

ranking.to_csv(result_path / "final_ranking.csv", index=False)

RANKING TỔNG HỢP (0.4×F1 + 0.3×AE + 0.3×UE)
                model prompt_strategy  f_score  ae  ue  composite_score
          llama3.1:8b             cot   0.0032 0.0 0.0           0.0013
          llama3.1:8b retrieval_aware   0.0020 0.0 0.0           0.0008
gemini-3.1-flash-lite             cot   0.0000 0.0 0.0           0.0000
gemini-3.1-flash-lite        few_shot   0.0000 0.0 0.0           0.0000
gemini-3.1-flash-lite       zero_shot   0.0000 0.0 0.0           0.0000
gemini-3.1-flash-lite retrieval_aware   0.0000 0.0 0.0           0.0000
         gpt-5.4-nano             cot   0.0000 0.0 0.0           0.0000
         gpt-5.4-nano        few_shot   0.0000 0.0 0.0           0.0000
         gpt-5.4-nano       zero_shot   0.0000 0.0 0.0           0.0000
         gpt-5.4-nano retrieval_aware   0.0000 0.0 0.0           0.0000
          llama3.1:8b        few_shot   0.0000 0.0 0.0           0.0000
          llama3.1:8b       zero_shot   0.0000 0.0 0.0           0.0000


## Push lên Git

In [35]:
%cd /content/smart-tourism

/content/smart-tourism


In [47]:
!git remote -v
!git status

origin	https://github.com/DuongThai2712/smart-tourism (fetch)
origin	https://github.com/DuongThai2712/smart-tourism (push)
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   Datasets/Results/cost_latency_summary.csv
	new file:   Datasets/Results/final_ranking.csv
	new file:   Datasets/Results/llm_results_v2.csv
	new file:   Datasets/Results/llm_results_v2.json
	new file:   Datasets/Results/summary_city_model.csv
	new file:   Datasets/Results/summary_model_strategy.csv



In [48]:
!git add Datasets/Results

In [38]:
!git pull origin main

From https://github.com/DuongThai2712/smart-tourism
 * branch            main       -> FETCH_HEAD
Already up to date.


In [49]:
!git commit -m "Add results"

[main 2172887] Add results
 6 files changed, 52882 insertions(+)
 create mode 100644 Datasets/Results/cost_latency_summary.csv
 create mode 100644 Datasets/Results/final_ranking.csv
 create mode 100644 Datasets/Results/llm_results_v2.csv
 create mode 100644 Datasets/Results/llm_results_v2.json
 create mode 100644 Datasets/Results/summary_city_model.csv
 create mode 100644 Datasets/Results/summary_model_strategy.csv


In [ ]:
!git config --global user.email "uit.edu.vn"
!git config --global user.name DuongThai2712

In [ ]:
!git push https://DuongThai2712: @github.com/DuongThai2712/smart-tourism.git main

Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 12 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (10/10), 403.59 KiB | 5.17 MiB/s, done.
Total 10 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/DuongThai2712/smart-tourism.git
   4b5a35e..2172887  main -> main
